In [16]:
import warnings
import pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    classification_report,
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
TARGET_COLUMN = "Class/ASD"
DATA_PATH = "../data/processed/autism_processed.csv"

In [17]:
df = pd.read_csv(DATA_PATH)
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Dataset shape: {df.shape}")
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print("Class distribution (ratio):")
print(y.value_counts(normalize=True).rename("ratio"))

Dataset shape: (800, 20)
Train shape: (640, 19), Test shape: (160, 19)
Class distribution (ratio):
Class/ASD
0    0.79875
1    0.20125
Name: ratio, dtype: float64


## Baseline Models with Repeated Stratified K-Fold CV

SMOTE is applied inside each fold via an imbalanced-learn pipeline to reduce leakage risk.

In [18]:
models = {
    "Logistic Regression": ImbPipeline(steps=[
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
    ]),
    "KNN": ImbPipeline(steps=[
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier()),
    ]),
    "Random Forest": ImbPipeline(steps=[
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE)),
    ]),
    "Gradient Boosting": ImbPipeline(steps=[
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ]),
}

try:
    from xgboost import XGBClassifier

    models["XGBoost"] = ImbPipeline(steps=[
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss")),
    ])
    print("XGBoost is available and included.")
except Exception:
    print("XGBoost not available, continuing without it.")

cv_repeated = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
print("Using RepeatedStratifiedKFold: 5 folds x 5 repeats (25 total splits)")

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

baseline_rows = []
for name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv_repeated,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    baseline_rows.append({
        "Model": name,
        "CV Accuracy Mean": scores["test_accuracy"].mean(),
        "CV Precision Mean": scores["test_precision"].mean(),
        "CV Recall Mean": scores["test_recall"].mean(),
        "CV Recall Std": scores["test_recall"].std(),
        "CV F1 Mean": scores["test_f1"].mean(),
        "CV F1 Std": scores["test_f1"].std(),
        "CV ROC AUC Mean": scores["test_roc_auc"].mean(),
    })

baseline_df = pd.DataFrame(baseline_rows).sort_values(
    by=["CV Recall Mean", "CV F1 Mean", "CV ROC AUC Mean"],
    ascending=False,
).reset_index(drop=True)

baseline_df

XGBoost is available and included.
Using RepeatedStratifiedKFold: 5 folds x 5 repeats (25 total splits)


,Model,CV Accuracy Mean,CV Precision Mean,CV Recall Mean,CV Recall Std,CV F1 Mean,CV F1 Std,CV ROC AUC Mean
0,KNN,0.789375,0.488999,0.818338,0.082401,0.610660,0.052284,0.860969
1,Logistic Regression,0.812187,0.525498,0.815631,0.084251,0.636713,0.047503,0.880874
2,Gradient Boosting,0.827500,0.557400,0.734892,0.081200,0.631581,0.051431,0.879113
3,Random Forest,0.827812,0.561094,0.703754,0.081785,0.621731,0.047861,0.893032
4,XGBoost,0.812813,0.532611,0.607569,0.119357,0.563223,0.077201,0.868787


## Hyperparameter Tuning with Repeated Stratified K-Fold

Tune top models selected from baseline CV rankings.

In [19]:
param_distributions = {
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1, 10, 50, 100],
        "model__solver": ["liblinear", "lbfgs"],
        "model__class_weight": [None, "balanced"],
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9, 11, 13, 15],
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2],
    },
    "Random Forest": {
        "model__n_estimators": [100, 200, 300, 500],
        "model__max_depth": [None, 5, 10, 20, 30],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__class_weight": [None, "balanced"],
    },
    "Gradient Boosting": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "model__max_depth": [2, 3, 4, 5],
        "model__subsample": [0.7, 0.85, 1.0],
    },
}

if "XGBoost" in models:
    param_distributions["XGBoost"] = {
        "model__n_estimators": [100, 200, 300, 500],
        "model__max_depth": [3, 4, 5, 6, 8],
        "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "model__subsample": [0.7, 0.85, 1.0],
        "model__colsample_bytree": [0.7, 0.85, 1.0],
    }

top_models = baseline_df["Model"].head(min(3, len(baseline_df))).tolist()
print(f"Tuning top models: {top_models}")

cv_tuning = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)
print("Tuning CV: RepeatedStratifiedKFold (5 folds x 2 repeats)")

best_estimators = {}
tuning_rows = []

for model_name in top_models:
    pipeline = models[model_name]
    n_iter = 30 if model_name in ["Random Forest", "XGBoost"] else 20

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions[model_name],
        n_iter=n_iter,
        scoring="f1",
        cv=cv_tuning,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
    )

    search.fit(X_train, y_train)
    best_estimators[model_name] = search.best_estimator_

    tuning_rows.append({
        "Model": model_name,
        "Best CV F1": search.best_score_,
        "Best Params": search.best_params_,
    })

    print(f"{model_name}: best CV F1 = {search.best_score_:.4f}")

tuning_df = pd.DataFrame(tuning_rows).sort_values(by="Best CV F1", ascending=False).reset_index(drop=True)
tuning_df

Tuning top models: ['KNN', 'Logistic Regression', 'Gradient Boosting']
Tuning CV: RepeatedStratifiedKFold (5 folds x 2 repeats)
KNN: best CV F1 = 0.6504
Logistic Regression: best CV F1 = 0.6646
Gradient Boosting: best CV F1 = 0.6553


,Model,Best CV F1,Best Params
0,Logistic Regression,0.664623,"{'model__solver': 'lbfgs', 'model__class_weigh..."
1,Gradient Boosting,0.655262,"{'model__subsample': 0.7, 'model__n_estimators..."
2,KNN,0.650393,"{'model__weights': 'uniform', 'model__p': 1, '..."


## Holdout Evaluation and Best Model Selection

Use recall-first ranking, then F1, then ROC AUC.

In [20]:
evaluation_rows = []
predictions = {}
probabilities = {}

for model_name, model in best_estimators.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = None

    predictions[model_name] = y_pred
    probabilities[model_name] = y_prob

    row = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
    }

    if y_prob is not None:
        row["ROC AUC"] = roc_auc_score(y_test, y_prob)
        row["Log Loss"] = log_loss(y_test, y_prob)
    else:
        row["ROC AUC"] = np.nan
        row["Log Loss"] = np.nan

    evaluation_rows.append(row)

results_df = pd.DataFrame(evaluation_rows).sort_values(
    by=["Recall", "F1", "ROC AUC"],
    ascending=False,
).reset_index(drop=True)

results_df

,Model,Accuracy,Precision,Recall,F1,ROC AUC,Log Loss
0,KNN,0.77500,0.468750,0.9375,0.625000,0.883179,1.728061
1,Logistic Regression,0.81250,0.518519,0.8750,0.651163,0.877441,0.433517
2,Gradient Boosting,0.78125,0.468085,0.6875,0.556962,0.872681,0.448752


In [21]:
best_model_name = results_df.iloc[0]["Model"]
best_model = best_estimators[best_model_name]
y_best_pred = predictions[best_model_name]
y_best_prob = probabilities[best_model_name]

print(f"Selected best model (recall-first): {best_model_name}")
print("\nClassification report:")
print(classification_report(y_test, y_best_pred))

stability_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
stability_scores = cross_validate(
    best_model,
    X_train,
    y_train,
    cv=stability_cv,
    scoring={"recall": "recall", "f1": "f1", "roc_auc": "roc_auc"},
    n_jobs=-1,
    return_train_score=False,
)

print("Repeated CV stability (train split):")
print(f"Recall: {stability_scores['test_recall'].mean():.4f} +/- {stability_scores['test_recall'].std():.4f}")
print(f"F1:     {stability_scores['test_f1'].mean():.4f} +/- {stability_scores['test_f1'].std():.4f}")
print(f"AUC:    {stability_scores['test_roc_auc'].mean():.4f} +/- {stability_scores['test_roc_auc'].std():.4f}")

if y_best_prob is not None:
    threshold_rows = []
    for thr in np.linspace(0.2, 0.8, 25):
        y_thr = (y_best_prob >= thr).astype(int)
        threshold_rows.append({
            "Threshold": thr,
            "Precision": precision_score(y_test, y_thr, zero_division=0),
            "Recall": recall_score(y_test, y_thr, zero_division=0),
            "F1": f1_score(y_test, y_thr, zero_division=0),
        })

    threshold_df = pd.DataFrame(threshold_rows).sort_values(
        by=["F1", "Recall"],
        ascending=False,
    ).reset_index(drop=True)

    print("\nTop threshold settings:")
    display(threshold_df.head(10))
else:
    print("This model does not provide probability output for threshold tuning.")

Selected best model (recall-first): KNN

Classification report:
              precision    recall  f1-score   support

           0       0.98      0.73      0.84       128
           1       0.47      0.94      0.62        32

    accuracy                           0.78       160
   macro avg       0.72      0.84      0.73       160
weighted avg       0.88      0.78      0.80       160

Repeated CV stability (train split):
Recall: 0.9177 +/- 0.0519
F1:     0.6439 +/- 0.0400
AUC:    0.8813 +/- 0.0317

Top threshold settings:


,Threshold,Precision,Recall,F1
0,0.550,0.491803,0.93750,0.645161
1,0.575,0.491803,0.93750,0.645161
2,0.475,0.468750,0.93750,0.625000
3,0.500,0.468750,0.93750,0.625000
4,0.525,0.468750,0.93750,0.625000
5,0.750,0.533333,0.75000,0.623377
6,0.775,0.533333,0.75000,0.623377
7,0.800,0.533333,0.75000,0.623377
8,0.675,0.500000,0.78125,0.609756
9,0.700,0.500000,0.78125,0.609756


In [22]:
results_df.to_csv("../models/train_benchmark_results.csv", index=False)
with open("../models/train_best_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("Saved ../models/train_benchmark_results.csv")
print("Saved ../models/train_best_model.pkl")

Saved ../models/train_benchmark_results.csv
Saved ../models/train_best_model.pkl
